In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

class ExpenseTracker:
    def __init__(self, filename="expenses.csv"):
        self.filename = filename
        try:
            self.data = pd.read_csv(self.filename)
        except FileNotFoundError:
            self.data = pd.DataFrame(columns=["Date", "Amount", "Category", "Description"])
            self.data.to_csv(self.filename, index=False)

    def add_expense(self, date, amount, category, description):
        # Input validation
        if amount <= 0:
            print(" Error: Amount must be positive.")
            return
        try:
            datetime.strptime(date, "%Y-%m-%d")
        except ValueError:
            print(" Error: Date format should be YYYY-MM-DD.")
            return

        new_entry = pd.DataFrame([[date, amount, category, description]],
                                 columns=["Date", "Amount", "Category", "Description"])
        self.data = pd.concat([self.data, new_entry], ignore_index=True)
        self.data.to_csv(self.filename, index=False)
        print(" Expense added successfully!")

    def get_summary(self):
        if self.data.empty:
            print("No data available for summary.")
            return None

        total = self.data["Amount"].sum()
        average = self.data["Amount"].mean()
        category_summary = self.data.groupby("Category")["Amount"].sum()

        print(" Expense Summary:")
        print(f"Total Expenses: ₹{total:.2f}")
        print(f"Average Expense: ₹{average:.2f}")
        print("\nCategory-wise Spending:")
        print(category_summary)
        return {"total": total, "average": average, "by_category": category_summary.to_dict()}

    def filter_expenses(self, category=None, start_date=None, end_date=None, min_amount=None, max_amount=None):
        filtered = self.data.copy()

        if category:
            filtered = filtered[filtered["Category"].str.lower() == category.lower()]
        if start_date:
            filtered = filtered[filtered["Date"] >= start_date]
        if end_date:
            filtered = filtered[filtered["Date"] <= end_date]
        if min_amount:
            filtered = filtered[filtered["Amount"] >= min_amount]
        if max_amount:
            filtered = filtered[filtered["Amount"] <= max_amount]

        print(" Filtered Expenses:")
        print(filtered if not filtered.empty else "No matching records found.")
        return filtered

    def generate_report(self):
        if self.data.empty:
            print("No data available for report.")
            return

        self.data["Date"] = pd.to_datetime(self.data["Date"])
        self.data["Month"] = self.data["Date"].dt.to_period("M")

        monthly_spending = self.data.groupby("Month")["Amount"].sum()
        category_spending = self.data.groupby("Category")["Amount"].sum()

        # --- Visualization Section ---
        plt.figure(figsize=(12, 6))
        sns.barplot(x=category_spending.index, y=category_spending.values)
        plt.title("Spending by Category")
        plt.xlabel("Category")
        plt.ylabel("Total Amount")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 6))
        monthly_spending.plot(kind='line', marker='o')
        plt.title(" Monthly Spending Trend")
        plt.xlabel("Month")
        plt.ylabel("Total Spending (₹)")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 8))
        plt.pie(category_spending, labels=category_spending.index, autopct='%1.1f%%', startangle=140)
        plt.title(" Spending Distribution by Category")
        plt.tight_layout()
        plt.show()

        print("\n Report generated successfully!")

# --------------------------
# Interactive Console Program
# --------------------------
def main():
    tracker = ExpenseTracker()

    while True:
        print("\n===== SMART EXPENSE TRACKER =====")
        print("1 Add New Expense")
        print("2 View Summary")
        print("3 Filter Expenses")
        print("4 Generate Report (Charts)")
        print("5 Exit")

        choice = input("Enter your choice (1-5): ")

        if choice == "1":
            date = input("Enter date (YYYY-MM-DD): ")
            try:
                amount = float(input("Enter amount: "))
            except ValueError:
                print(" Invalid amount. Try again.")
                continue
            category = input("Enter category (e.g., Food, Travel, Rent): ")
            description = input("Enter description: ")
            tracker.add_expense(date, amount, category, description)

        elif choice == "2":
            tracker.get_summary()

        elif choice == "3":
            category = input("Enter category to filter (or leave blank): ") or None
            start_date = input("Enter start date (YYYY-MM-DD) or leave blank: ") or None
            end_date = input("Enter end date (YYYY-MM-DD) or leave blank: ") or None
            min_amount = input("Enter minimum amount or leave blank: ")
            max_amount = input("Enter maximum amount or leave blank: ")

            min_amount = float(min_amount) if min_amount else None
            max_amount = float(max_amount) if max_amount else None

            tracker.filter_expenses(category, start_date, end_date, min_amount, max_amount)

        elif choice == "4":
            tracker.generate_report()

        elif choice == "5":
            print(" Exiting... Goodbye!")
            break

        else:
            print(" Invalid choice. Please enter 1 - 5.")

if __name__ == "__main__":
    main()


Matplotlib is building the font cache; this may take a moment.



===== SMART EXPENSE TRACKER =====
1 Add New Expense
2 View Summary
3 Filter Expenses
4 Generate Report (Charts)
5 Exit


Enter your choice (1-5):  1
Enter date (YYYY-MM-DD):  2002-05-23
Enter amount:  10000
Enter category (e.g., Food, Travel, Rent):  Food
